<div style="background: linear-gradient(135deg, #1a1a2e, #16213e, #0f3460); border-radius:14px; padding:24px; text-align:center; border-left:5px solid #e94560;">
  <h1 style="color:#ffffff; font-size:28px; margin:0; letter-spacing:2px;">🌌 STELLAR CLASS — EDA</h1>
  <p style="color:#a0c4ff; font-size:14px; margin:8px 0 0;">Playground Series S6E6 · SDSS Spectral Classification · GALAXY / QSO / STAR</p>
</div>

In [ ]:
# ── Cell 1 · Setup & Quick Snapshot ───────────────────────────────────────────
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import kagglehub
path = kagglehub.competition_download('playground-series-s6e6')
df   = pd.read_csv(f'{path}/train.csv', index_col=0)

# ── Confirmed columns ─────────────────────────────────────────────────────────
# alpha, delta        → sky coordinates (RA / Dec)
# u, g, r, i, z       → SDSS photometric filter magnitudes
# redshift            → cosmological redshift
# spectral_type       → categorical
# galaxy_population   → categorical
# class               → target  (GALAXY / QSO / STAR)

NUM_COLS = ['alpha','delta','u','g','r','i','z','redshift']
PALETTE  = {'GALAXY':'#5aab61', 'QSO':'#9b72cf', 'STAR':'#e8a838'}
CLASS_ORDER = ['GALAXY','QSO','STAR']

print(f"Shape  : {df.shape}")
print(f"Nulls  : {df.isnull().sum().sum()}")
print(f"Classes: {df['class'].value_counts().to_dict()}")
df.describe().round(3)

In [ ]:
# ── Cell 2 · Class Distribution ───────────────────────────────────────────────
class_counts = df['class'].value_counts().reindex(CLASS_ORDER)
colors       = [PALETTE[c] for c in CLASS_ORDER]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.patch.set_facecolor('#f7f7f7')

# --- Bar chart ----------------------------------------------------------------
ax = axes[0]
ax.set_facecolor('#f0f0f0')
bars   = ax.bar(CLASS_ORDER, class_counts.values, color=colors,
                edgecolor='white', linewidth=1.2, width=0.5)
offset = class_counts.max() * 0.015
for bar, val in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + offset,
            f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold', color='#333')
ax.set_title('Class Count', fontsize=13, fontweight='bold', pad=10, color='#222')
ax.set_ylabel('Count')
ax.spines[['top','right']].set_visible(False)
ax.tick_params(labelsize=10)

# --- Pie chart ----------------------------------------------------------------
ax2 = axes[1]
ax2.set_facecolor('#f0f0f0')
wedges, texts, autotexts = ax2.pie(
    class_counts.values, labels=CLASS_ORDER, colors=colors,
    autopct='%1.1f%%', startangle=140, pctdistance=0.75,
    wedgeprops=dict(edgecolor='white', linewidth=1.8)
)
for t in autotexts: t.set_fontsize(10); t.set_color('#111')
ax2.set_title('Class Share (%)', fontsize=13, fontweight='bold', pad=10, color='#222')

plt.suptitle('Dataset is imbalanced — GALAXY dominates at ~65%', fontsize=11, color='#555', y=1.01)
plt.tight_layout(); plt.show()
print("\n💡 Use stratified splits & class-weight aware models (e.g. class_weight='balanced') to handle imbalance.")

In [ ]:
# ── Cell 3 · 🌟 Floating Range Bar — Feature Values per Class ─────────────────
# Each bar = class mean.  Floating vertical line = min–max range in that class.
# Short line → tight cluster (consistent feature). Long line → high spread.
# Tall bar differences → feature SEPARATES classes well (high ML value).

plot_features = ['u', 'g', 'r', 'redshift', 'i', 'z']
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.patch.set_facecolor('#f7f7f7')
axes = axes.flatten()

for idx, feat in enumerate(plot_features):
    ax  = axes[idx]
    ax.set_facecolor('#efefef')

    stats = (df.groupby('class')[feat]
               .agg(['mean','min','max'])
               .reindex(CLASS_ORDER))
    x          = np.arange(len(CLASS_ORDER))
    bar_colors = [PALETTE[c] for c in CLASS_ORDER]

    # Bars = mean
    bars = ax.bar(x, stats['mean'], color=bar_colors, edgecolor='white',
                  linewidth=1.0, width=0.5, alpha=0.88, zorder=2)

    # Floating lines = min–max range
    for i, cls in enumerate(CLASS_ORDER):
        lo, hi, mu = stats.loc[cls, 'min'], stats.loc[cls, 'max'], stats.loc[cls, 'mean']
        ax.plot([i, i],         [lo, hi], color='#333', linewidth=2.2, zorder=3, solid_capstyle='round')
        ax.plot([i-.13, i+.13], [lo, lo], color='#333', linewidth=2.2, zorder=4)
        ax.plot([i-.13, i+.13], [hi, hi], color='#333', linewidth=2.2, zorder=4)
        ax.scatter(i, mu, color='white', edgecolors='#333', s=50, zorder=5, linewidths=1.6)

    # Mean labels
    for bar, val in zip(bars, stats['mean']):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + abs(bar.get_height()) * 0.02 + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=7.5,
                color='#222', fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_ORDER, fontsize=9)
    ax.set_title(f'{feat}  ·  mean (bar)  +  min–max range (line)',
                 fontsize=10, fontweight='bold', color='#222', pad=7)
    ax.set_ylabel(feat, fontsize=9)
    ax.spines[['top','right']].set_visible(False)
    ax.tick_params(axis='y', labelsize=8)

legend_patches = [mpatches.Patch(color=PALETTE[c], label=c) for c in CLASS_ORDER]
fig.legend(handles=legend_patches, loc='lower center', ncol=3,
           fontsize=10, frameon=True, framealpha=0.85, bbox_to_anchor=(0.5, -0.03))
plt.suptitle('Per-Feature: Mean per Class  ●  Floating Lines = Full Value Range',
             fontsize=13, fontweight='bold', color='#222', y=1.02)
plt.tight_layout(); plt.show()

print("\n💡 redshift is the star — QSOs sit at very high redshift vs STARS near zero.")
print("   Magnitude bands (u,g,r,i,z) separate QSOs from GALAXYs and STARs.")

In [ ]:
# ── Cell 4 · Redshift Distribution — The #1 Discriminating Feature ────────────
# Redshift alone almost perfectly separates STARs (z≈0) from QSOs (z>>1).
# This cell makes that dramatically visible.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#f7f7f7')

# --- KDE plot (full range) ---
ax = axes[0]
ax.set_facecolor('#ececec')
for cls in CLASS_ORDER:
    sub = df[df['class'] == cls]['redshift']
    sub.plot.kde(ax=ax, color=PALETTE[cls], label=cls, linewidth=2.2)
ax.set_title('Redshift Distribution per Class (KDE)', fontsize=11, fontweight='bold', color='#222')
ax.set_xlabel('Redshift', fontsize=10)
ax.set_ylabel('Density', fontsize=10)
ax.legend(fontsize=9, framealpha=0.85)
ax.spines[['top','right']].set_visible(False)

# --- Box plot (log scale to handle QSO outliers) ---
ax2 = axes[1]
ax2.set_facecolor('#ececec')
data_by_class = [df[df['class'] == c]['redshift'].dropna().values for c in CLASS_ORDER]
bp = ax2.boxplot(data_by_class, patch_artist=True, notch=False,
                 medianprops=dict(color='white', linewidth=2.0),
                 whiskerprops=dict(linewidth=1.4, color='#444'),
                 capprops=dict(linewidth=1.4, color='#444'),
                 flierprops=dict(marker='.', markersize=1.5, alpha=0.3, color='#888'))
for patch, cls in zip(bp['boxes'], CLASS_ORDER):
    patch.set_facecolor(PALETTE[cls]); patch.set_alpha(0.85)
ax2.set_xticks([1,2,3]); ax2.set_xticklabels(CLASS_ORDER, fontsize=10)
ax2.set_title('Redshift Box Plot per Class', fontsize=11, fontweight='bold', color='#222')
ax2.set_ylabel('Redshift', fontsize=10)
ax2.spines[['top','right']].set_visible(False)

plt.suptitle('Redshift separates STARs (≈0) and GALAXYs (low) from QSOs (high)',
             fontsize=11, color='#555', y=1.01)
plt.tight_layout(); plt.show()

print("\n💡 STAR redshift ≈ 0 (within Milky Way).")
print("   GALAXY redshift ~ 0.1–0.5 (nearby universe).")
print("   QSO redshift can exceed 3–5 (distant quasars) → dominant discriminating feature.")

In [ ]:
# ── Cell 5 · Correlation Heatmap + Mutual Information ─────────────────────────
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.patch.set_facecolor('#f7f7f7')

# --- Correlation heatmap ------------------------------------------------------
ax = axes[0]
corr = df[NUM_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=ax, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 9, 'weight': 'bold'},
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix\n(lower triangle)', fontsize=11,
             fontweight='bold', color='#222', pad=8)
ax.tick_params(axis='x', rotation=30, labelsize=9)
ax.tick_params(axis='y', rotation=0,  labelsize=9)

# --- Mutual Information -------------------------------------------------------
ax2 = axes[1]
ax2.set_facecolor('#f0f0f0')

X  = df[NUM_COLS].fillna(0)
le = LabelEncoder()
y  = le.fit_transform(df['class'])
mi = mutual_info_classif(X, y, random_state=42)
mi_s = pd.Series(mi, index=NUM_COLS).sort_values(ascending=True)

bar_mi_colors = plt.cm.viridis(np.linspace(0.25, 0.85, len(mi_s)))
bars = ax2.barh(mi_s.index, mi_s.values, color=bar_mi_colors,
                edgecolor='white', linewidth=0.8, height=0.55)
for bar, val in zip(bars, mi_s.values):
    ax2.text(val + 0.003, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=9, fontweight='bold', color='#222')
ax2.set_title('Mutual Information with Target\n(higher = more predictive power)',
              fontsize=11, fontweight='bold', color='#222', pad=8)
ax2.set_xlabel('Mutual Information Score', fontsize=9)
ax2.spines[['top','right']].set_visible(False)
ax2.tick_params(labelsize=9)

plt.suptitle('Which features carry the most signal?', fontsize=11, color='#555', y=1.02)
plt.tight_layout(); plt.show()

print("\n💡 High MI = feature strongly informs the class label → prioritise in model.")
print("   Highly correlated pairs (|r| > 0.8) are redundant → keep the one with higher MI.")

In [ ]:
# ── Cell 6 · Spectral Type & Galaxy Population Breakdown ──────────────────────
# Categorical features can encode class membership directly.

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.patch.set_facecolor('#f7f7f7')

# --- Spectral type × class heatmap -------------------------------------------
ax = axes[0]
ct = pd.crosstab(df['spectral_type'], df['class'])
ct = ct.reindex(columns=CLASS_ORDER, fill_value=0)
ct = ct.loc[ct.sum(axis=1) > 0]    # drop empty rows
sns.heatmap(ct, ax=ax, cmap='YlOrBr', annot=True, fmt='d',
            linewidths=0.4, linecolor='white',
            annot_kws={'size': 8},
            cbar_kws={'shrink': 0.8})
ax.set_title('Spectral Type  ×  Class\n(counts)', fontsize=11, fontweight='bold', color='#222', pad=8)
ax.set_xlabel('Class', fontsize=9); ax.set_ylabel('Spectral Type', fontsize=9)
ax.tick_params(axis='x', rotation=0,  labelsize=9)
ax.tick_params(axis='y', rotation=0,  labelsize=8)

# --- Galaxy population stacked bar -------------------------------------------
ax2 = axes[1]
ax2.set_facecolor('#f0f0f0')
top_pops  = df['galaxy_population'].value_counts().head(6).index.tolist()
gp_ct     = pd.crosstab(df['class'], df['galaxy_population'])
# keep only top populations; fill missing
for col in top_pops:
    if col not in gp_ct.columns:
        gp_ct[col] = 0
gp_ct     = gp_ct[top_pops].reindex(CLASS_ORDER, fill_value=0)
gp_pct    = gp_ct.div(gp_ct.sum(axis=1), axis=0) * 100

bar_colors = ['#e05a5a','#e8a838','#7ec8e3','#5aab61','#9b72cf','#c27c3a']
bottom = np.zeros(len(CLASS_ORDER))
for i, pop in enumerate(top_pops):
    ax2.bar(CLASS_ORDER, gp_pct[pop], bottom=bottom,
            label=str(pop), color=bar_colors[i], edgecolor='white', linewidth=0.8)
    bottom += gp_pct[pop].values

ax2.set_title('Galaxy Population Composition\nper Class (% of top 6 populations)',
              fontsize=11, fontweight='bold', color='#222', pad=8)
ax2.set_ylabel('Percentage (%)', fontsize=9)
ax2.spines[['top','right']].set_visible(False)
ax2.tick_params(labelsize=9)
ax2.legend(title='Population', fontsize=8, title_fontsize=8,
           loc='upper right', framealpha=0.85)

plt.suptitle('Categorical features: Spectral Type & Galaxy Population encode real structure',
             fontsize=11, color='#555', y=1.02)
plt.tight_layout(); plt.show()

print("\n💡 If spectral_type maps exclusively to one class → near-zero entropy → very strong feature.")
print("   Encode as ordinal (not one-hot) to keep physical ordering and reduce dimensionality.")